<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/15_triangle_recall_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys, os
from pathlib import Path

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / "src").is_dir():
                return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

print("Project root set to:", PROJ_ROOT)
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

Project root set to: /content/regime-aware-ml-trading/regime-aware-ml-trading
src exists: True


# 15 — Triangle Detector: Rejection-Funnel Diagnosis and Recall Ablation

**Objective:** The supervisor's stated priority was to improve detector quality
before further ML work, having flagged channel detection as unconvincing.
Having addressed channels (see notebook 08), this notebook applies the same
evidence-based diagnostic methodology to the triangle detector, whose recall
(12 detections over 15 years) is too sparse for reliable downstream ML.

**Approach:** rather than blindly loosening parameters to increase count,
a rejection-funnel diagnostic first identifies exactly which validation gate
is the true bottleneck, and only that gate is targeted for ablation.

In [2]:
from src.data.load_data import load_spy
df_yf = load_spy(source='csv')
print(f'Loaded {len(df_yf)} bars')

[*********************100%***********************]  1 of 1 completed

Saved 4024 rows to /content/regime-aware-ml-trading/regime-aware-ml-trading/data/raw/spy.csv
Loaded 4024 bars


In [3]:
import inspect
from src.patterns.triangles import detect_triangle_pattern

print(inspect.getsource(detect_triangle_pattern))

def detect_triangle_pattern(df, window=20, min_convergence_pct=0.05,
                            cooldown=10, return_details=False,
                            pivot_order=2, min_pivots=2, min_r=0.85,
                            flat_threshold_mult=0.25):
    """Detect triangle patterns using the pivot + linregress approach.

    Closely follows the *TrianglePricePatterns* reference notebook:

    1. Identify swing highs / lows (±3-bar neighbourhood).
    2. ``linregress`` on pivot points → slope, intercept, *r*.
    3. Require ``|r| >= min_r`` on each trendline (tight pivot alignment).
    4. Classify by slope signs: ascending / descending / symmetric.
    5. Fire signal when current bar breaks out of the recent range.

    **Signal localization (yellow diamond):**
    The signal bar is the breakout bar — the bar whose High exceeds the
    recent 3-bar high by 0.3×ATR (upside breakout) or whose Low drops
    below the recent 3-bar low by 0.3×ATR (downside breakout).  For
    descendin

In [4]:
import subprocess
result = subprocess.run(
    ['git', '-C', '/content/regime-aware-ml-trading', 'pull', 'origin', 'main'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

with open('/content/regime-aware-ml-trading/regime-aware-ml-trading/src/patterns/triangles.py') as f:
    content = f.read()
print("flat_threshold_mult found in Colab's copy:", 'flat_threshold_mult' in content)

import sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.patterns.triangles import detect_triangle_pattern
import inspect
print(inspect.signature(detect_triangle_pattern))

Already up to date.

From https://github.com/zaetae/regime-aware-ml-trading
 * branch            main       -> FETCH_HEAD

flat_threshold_mult found in Colab's copy: True
(df, window=20, min_convergence_pct=0.05, cooldown=10, return_details=False, pivot_order=2, min_pivots=2, min_r=0.85, flat_threshold_mult=0.25)


In [6]:
import pandas as pd
import numpy as np
_, tri_details_final = detect_triangle_pattern(df_yf.copy(), return_details=True)

type_counts_final = pd.Series([d['pattern_type'] for d in tri_details_final]).value_counts()
print(f"Total triangles (final, both fixes applied): {len(tri_details_final)}")
print(type_counts_final)

Total triangles (final, both fixes applied): 34
symmetric_triangle          15
ascending_triangle          13
desc_triangle_upper_test     4
descending_triangle          2
Name: count, dtype: int64
